# Bolt #6: Residual Analysis & Reporting

**Objective**: Analyze the Hybrid Baseline residuals to identify performance bottlenecks and validate statistical assumptions.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as g_o
from pathlib import Path
import scipy.stats as stats
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import acf, pacf

# Configuration
RUN_PATH = Path("../../artifacts/runs/dry_run_v1_retry3/")
RAW_DATA_PATH = Path("../../data/raw/")

print(f"Analyzing run: {RUN_PATH.resolve().name}")

## 1. Data Loading & Enrichment

Joining OOF residuals with Oil prices and Holiday events.

In [ ]:
# Load OOF Residuals
df_oof = pd.read_csv(RUN_PATH / "oof_residuals.csv", parse_dates=["date"])

# Load Oil
df_oil = pd.read_csv(RAW_DATA_PATH / "oil.csv", parse_dates=["date"])
# Axiom: Forward-fill oil prices (weekends/holidays)
df_oil = df_oil.set_index("date").resample('D').ffill().reset_index()

# Load Holidays
df_holidays = pd.read_csv(RAW_DATA_PATH / "holidays_events.csv", parse_dates=["date"])

# Merge
df = df_oof.merge(df_oil, on="date", how="left")
df = df.merge(df_holidays, on="date", how="left")

print(f"Enriched Data Shape: {df.shape}")
df.head()

## 2. Segment Metrics & Macro Performance

Identifying high-level failure patterns across the Store-Family matrix.

In [ ]:
# Calculate RMSLE per segment
df['log_pred'] = np.log1p(np.maximum(0, df['sales_pred']))
df['log_actual'] = np.log1p(df['sales'])
df['sq_log_error'] = (df['log_pred'] - df['log_actual'])**2

agg_metrics = df.groupby(['store_nbr', 'family']).agg({
    'sq_log_error': 'mean',
    'sales': ['sum', 'mean'],
    'residual': 'std'
}).reset_index()

agg_metrics.columns = ['store_nbr', 'family', 'rmsle', 'total_sales', 'avg_sales', 'residual_std']
agg_metrics['rmsle'] = np.sqrt(agg_metrics['rmsle'])

print(f"Average Segment RMSLE: {agg_metrics['rmsle'].mean():.4f}")
agg_metrics.sort_values('rmsle', ascending=False).head(10)

In [ ]:
# Macro Heatmap: Matrix of RMSLE
fig = px.density_heatmap(
    agg_metrics, 
    x="family", 
    y="store_nbr", 
    z="rmsle",
    title="Global RMSLE Heatmap (Store vs Family)",
    labels={'rmsle': 'RMSLE'},
    color_continuous_scale="Viridis",
    nbinsy=agg_metrics['store_nbr'].nunique()
)
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

## 3. Statistical Residual Analysis

Validating if the residuals follow theoretical assumptions (Normality, Independence).

In [ ]:
# Normality Tests
residuals_sample = df['residual'].dropna().sample(min(5000, len(df)))
k2, p_k2 = stats.normaltest(residuals_sample)
shapiro_stat, p_shapiro = stats.shapiro(residuals_sample[:1000])

print("--- Normality Tests ---")
print(f"D'Agostino's K^2 p-value: {p_k2:.4e}")
print(f"Shapiro-Wilk p-value (sample=1000): {p_shapiro:.4e}")

fig_dist = px.histogram(df, x="residual", marginal="box", title="Distribution of Residuals")
fig_dist.show()

In [ ]:
# Autocorrelation (Durbin-Watson)
# DW near 2 means no autocorrelation. Values < 2 suggest positive correlation.
dw_stat = durbin_watson(df['residual'].dropna())
print(f"Durbin-Watson Statistic: {dw_stat:.4f}")

if dw_stat < 1.5 or dw_stat > 2.5:
    print("WARNING: Significant autocorrelation detected in residuals. Seasonal signals likely remain.")
else:
    print("Residuals appear largely independent.")